Qiskit

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

qr = QuantumRegister(3)
cr = ClassicalRegister(2)
qc = QuantumCircuit(qr, cr)

# Setup & Alice's Ops (Standard)
qc.h(qr[1])
qc.cx(qr[1], qr[2])
qc.h(qr[0])
qc.cx(qr[0], qr[1])
qc.h(qr[0])

# Measure
qc.measure(qr[0], cr[0])
qc.measure(qr[1], cr[1])

# Step 5: Bob's conditional gates
with qc.if_test((cr[1], 1)):
    qc.x(qr[2])
with qc.if_test((cr[0], 1)):
    qc.z(qr[2])

Cirq

In [6]:
import cirq

q = cirq.LineQubit.range(3)
circuit = cirq.Circuit()

# Step 1 & 2: Bell Pair and Source State
circuit.append([cirq.H(q[1]), cirq.CNOT(q[1], q[2])])
circuit.append(cirq.H(q[0]))

# Step 3: Alice's operations
circuit.append([cirq.CNOT(q[0], q[1]), cirq.H(q[0])])

# Step 4: Measure
circuit.append([cirq.measure(q[0], key='m0'), cirq.measure(q[1], key='m1')])

# Step 5: Bob's conditional logic (Fixed)
# We apply the X and Z gates conditioned on the classical measurement keys
circuit.append(cirq.X(q[2]).with_classical_controls('m1'))
circuit.append(cirq.Z(q[2]).with_classical_controls('m0'))

Pennylane

In [4]:
import pennylane as qml

dev = qml.device("default.qubit", wires=3)

@qml.qnode(dev)
def teleportation():
    # Setup: Entangle Alice's second qubit and Bob's qubit
    qml.Hadamard(wires=1)
    qml.CNOT(wires=[1, 2])
    
    # Source: Prepare the state to be teleported (e.g., Hadamard)
    qml.Hadamard(wires=0)

    # Alice's local operations
    qml.CNOT(wires=[0, 1])
    qml.Hadamard(wires=0)

    # Mid-circuit Measurement (The "Alice" part)
    m0 = qml.measure(0)
    m1 = qml.measure(1)

    # Bob's Conditionals (The "Classical Feedback" part)
    qml.cond(m1, qml.PauliX)(2)
    qml.cond(m0, qml.PauliZ)(2)

    return qml.state()